# Hugging Face models

SDXL Turbo, Stable Diffusion XL, the SDXL refiner, SpeechT5, and FLUX.1-schnell. A CUDA GPU is required. FLUX is gated and needs a large GPU. Images and the wav file are written in this folder.

These cells match `transformers==4.56.2`, `diffusers==0.32.2`, and `datasets==3.6.0`. Put `HF_TOKEN` in the repo `.env`. FLUX also needs access granted on the model page.


## Setup

In [1]:
import os
from pathlib import Path


In [ ]:
import torch
from dotenv import load_dotenv

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
day_dir = repo_root / "W3-Open Source Models" / "Day-1"
load_dotenv(repo_root / ".env", override=True)

hf_home = repo_root / "W3-Open Source Models" / "hf-cache"
hf_home.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(hf_home)
os.environ["HF_HUB_CACHE"] = str(hf_home / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_home / "datasets")

from huggingface_hub import login
from IPython.display import display

login(os.getenv("HF_TOKEN"))
print("Model cache:", hf_home)

if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU")

device = "cuda"
dtype = torch.float16
print(torch.cuda.get_device_name(0))


## SDXL Turbo

Four steps. This is the small, fast model.

In [ ]:
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=dtype,
    variant="fp16",
)
pipe.to(device)

prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
image.save(day_dir / "sdxl-turbo.png")
display(image)

del pipe
torch.cuda.empty_cache()


## Stable Diffusion XL

The base model, thirty steps.

In [ ]:
from diffusers import DiffusionPipeline

pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=dtype,
    use_safetensors=True,
    variant="fp16",
)
pipe.to(device)

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"
image = pipe(prompt=prompt, num_inference_steps=30).images[0]
image.save(day_dir / "sdxl-base.png")
display(image)

del pipe
torch.cuda.empty_cache()


## Base and refiner

The base runs the first 80% of the steps. The refiner runs the rest. They share the second text encoder and the VAE.

In [ ]:
from diffusers import DiffusionPipeline

base = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=dtype,
    variant="fp16",
    use_safetensors=True,
)
base.to(device)
refiner = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    text_encoder_2=base.text_encoder_2,
    vae=base.vae,
    torch_dtype=dtype,
    use_safetensors=True,
    variant="fp16",
)
refiner.to(device)

n_steps = 40
high_noise_frac = 0.8
prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"

latents = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent",
).images
image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=latents,
).images[0]
image.save(day_dir / "sdxl-refined.png")
display(image)

del base, refiner
torch.cuda.empty_cache()


## Speech

SpeechT5, with one speaker embedding from the CMU Arctic set.

In [ ]:
import soundfile as sf
from datasets import load_dataset
from IPython.display import Audio
from transformers import pipeline

synthesizer = pipeline("text-to-speech", "microsoft/speecht5_tts", device=device)
embeddings = load_dataset(
    "matthijs/cmu-arctic-xvectors",
    split="validation",
    trust_remote_code=True,
)
speaker_embedding = torch.tensor(embeddings[7306]["xvector"]).unsqueeze(0)
speech = synthesizer(
    "Hi to an artificial intelligence engineer, on the way to mastery!",
    forward_params={"speaker_embeddings": speaker_embedding},
)

path = day_dir / "speech.wav"
sf.write(path, speech["audio"], speech["sampling_rate"])
display(Audio(path))

del synthesizer
torch.cuda.empty_cache()


## FLUX.1-schnell

Gated. Accept the terms at [black-forest-labs/FLUX.1-schnell](https://huggingface.co/black-forest-labs/FLUX.1-schnell), then run this. Four steps, and it wants a large GPU.

In [ ]:
import time

from diffusers import FluxPipeline

start = time.perf_counter()
pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=torch.bfloat16,
).to(device)
generator = torch.Generator(device=device).manual_seed(0)
prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"
image = pipe(
    prompt,
    guidance_scale=0.0,
    num_inference_steps=4,
    max_sequence_length=256,
    generator=generator,
).images[0]
image.save(day_dir / "flux-schnell.png")
display(image)
print(f"This took {time.perf_counter() - start:.1f} seconds")

del pipe
torch.cuda.empty_cache()
